In [1]:
# MULTIPANEL FIGURE GENERATION

# This notebook:
# 1. Loads pre-computed departures, percentages, and statistics from NetCDF
# 2. Creates the final multipanel precipitation figure
#
# Input: NetCDF files from multipanel_percentile/multipanel_{RCP}/
# Output: **Figure 3** High-resolution multipanel figure (PNG/PDF)

import os
import numpy as np
import xarray as xr
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.offsetbox import AnchoredText
from matplotlib.colors import ListedColormap
from matplotlib import gridspec, colors, colormaps
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cartopy.io.shapereader as shpreader

# Force non-interactive backend
matplotlib.use('Agg')


In [10]:
# CONFIGURATION

DATA_ROOT = "/path/to/SDHI_Precip_supporting_data"  # <-- EDIT THIS LINE, if notebook is in root you can use "./"

RCP_scenario = '8p5'  # Must match the scenario used in notebook 1 where we prepared tha data to plot (file naming)
percentile_band = '99th-100th'# Again, needs to match the band (percipitation interval) names in the propr notebook for data prep

# Input directory (from notebook 1)
input_dir = f'{DATA_ROOT}/multipanel_percentile/multipanel_{RCP_scenario}'

# Output settings
output_dir = f'{DATA_ROOT}/figures'
os.makedirs(output_dir, exist_ok=True)

figure_basename = f'multipanel_99th_percentile_EOC{RCP_scenario}'

# Visualization parameters
alpha = 0.05  # Significance threshold
sig_display_threshold = 0.25  # Fraction of cells that must be significant to show stippling
downsampling_factor = (21, 21)  # Coarsen grid for readability

print("Configuration:")
print(f"  RCP Scenario: {RCP_scenario}")
print(f"  Input directory: {input_dir}")
print(f"  Output directory: {output_dir}")
print(f"  Significance threshold: α = {alpha}")
print(f"  Stippling threshold: {sig_display_threshold}")
print(f"  Downsampling: {downsampling_factor}")

Configuration:
  RCP Scenario: 8p5
  Input directory: ./multipanel_percentile/multipanel_8p5
  Output directory: ./figures
  Significance threshold: α = 0.05
  Stippling threshold: 0.25
  Downsampling: (21, 21)


In [4]:
# UTILITY FUNCTIONS

def downsample_ds(ds, aggregation_factor=(21, 21), reducer='mean'):
    """Downsample dataset for faster plotting"""
    boundary = "trim"
    
    coarsened = ds.coarsen(
        west_east=aggregation_factor[0], 
        south_north=aggregation_factor[1], 
        boundary=boundary
    )
    
    if reducer == 'mean':
        return coarsened.mean(skipna=True)
    elif reducer == 'min':
        return coarsened.min(skipna=True)
    elif reducer == 'max':
        return coarsened.max(skipna=True)
    else:
        raise ValueError(f"Unknown reducer: {reducer}")


def truncate_colormap(cmap, minval=0.0, maxval=1.0, n=100):
    """Create a subset of an input colormap"""
    new_cmap = colors.LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap


def discrete_cmap(N, base_cmap=None):
    """Create an N-bin discrete colormap"""
    base = plt.cm.get_cmap(base_cmap) if isinstance(base_cmap, str) else base_cmap
    color_list = base(np.linspace(0, 1, N))
    cmap_name = base.name + str(N)
    return base.from_list(cmap_name, color_list, N)


def create_discretized_colormap_with_arrows_and_ticks(cmap_name, num_bins, vmin, vmax):
    """Create a discretized colormap with over/under arrows"""
    cmap = plt.get_cmap(cmap_name) if isinstance(cmap_name, str) else cmap_name
    colors_list = cmap(np.linspace(0, 1, num_bins))
    
    colormap_with_arrows = ListedColormap(colors_list[1:-1])
    colormap_with_arrows.set_over(colors_list[-1])
    colormap_with_arrows.set_under(colors_list[0])
    
    tick_positions = np.linspace(vmin, vmax, num_bins - 1)
    return colormap_with_arrows, tick_positions

In [5]:
# LOAD DATA

print("LOADING DATA")


seasons = ['annual', 'mam', 'jja', 'son', 'djf']

# Load all datasets
departures = {}
percentages = {}
hist_means = {}
statistics = {}

print("\nLoading datasets...")
for season in seasons:
    print(f"\n  Loading {season.upper()}...")
    
    # Load departures
    departures[season] = xr.open_dataset(f"{input_dir}/departures/{season}_departure.nc")
    print(f"    Departure variables: {list(departures[season].data_vars)}")
    print(f"    Departure dims: {dict(departures[season].dims)}")
    
    # Load percentages
    percentages[season] = xr.open_dataset(f"{input_dir}/percentages/{season}_percentage.nc")
    print(f"    Percentage variables: {list(percentages[season].data_vars)}")
    
    # Load historical means
    hist_means[season] = xr.open_dataset(f"{input_dir}/hist_means/{season}_hist_mean.nc")
    print(f"    Hist mean variables: {list(hist_means[season].data_vars)}")
    
    # Check if data exists and has values
    first_var = list(hist_means[season].data_vars)[0]
    data_sample = hist_means[season][first_var]
    print(f"    Hist mean data check:")
    print(f"      Variable: {first_var}")
    print(f"      Shape: {data_sample.shape}")
    print(f"      Min: {data_sample.min().values}")
    print(f"      Max: {data_sample.max().values}")
    print(f"      Has NaN: {np.isnan(data_sample.values).any()}")
    print(f"      All NaN: {np.isnan(data_sample.values).all()}")
    
    # Load statistics
    statistics[season] = xr.open_dataset(f"{input_dir}/statistics/{season}_mwu_fdr.nc")
    print(f"    Statistics variables: {list(statistics[season].data_vars)}")

print("\nAll data loaded!")

LOADING DATA

Loading datasets...

  Loading ANNUAL...


/anaconda3/envs/pyMET/lib/python3.10/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'cfradial1' loading failed:
/anaconda3/envs/pyMET/lib/python3.10/site-packages/h5py/defs.cpython-310-x86_64-linux-gnu.so: undefined symbol: H5Pget_fapl_direct
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)
/anaconda3/envs/pyMET/lib/python3.10/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'furuno' loading failed:
/anaconda3/envs/pyMET/lib/python3.10/site-packages/h5py/defs.cpython-310-x86_64-linux-gnu.so: undefined symbol: H5Pget_fapl_direct
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)
/anaconda3/envs/pyMET/lib/python3.10/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'gamic' loading failed:
/anaconda3/envs/pyMET/lib/python3.10/site-packages/h5py/defs.cpython-310-x86_64-linux-gnu.so: undefined symbol: H5Pget_fapl_direct
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWa

    Departure variables: ['PRECIP_PERIOD']
    Departure dims: {'south_north': 899, 'west_east': 1399}
    Percentage variables: ['PRECIP_PERIOD']
    Hist mean variables: ['PRECIP_PERIOD']
    Hist mean data check:
      Variable: PRECIP_PERIOD
      Shape: (899, 1399)
      Min: 0.0
      Max: 61.46666666666667
      Has NaN: False
      All NaN: False
    Statistics variables: ['p_value', 'p_value_fdr_corrected']

  Loading MAM...
    Departure variables: ['PRECIP_PERIOD']
    Departure dims: {'south_north': 899, 'west_east': 1399}
    Percentage variables: ['PRECIP_PERIOD']
    Hist mean variables: ['PRECIP_PERIOD']
    Hist mean data check:
      Variable: PRECIP_PERIOD
      Shape: (899, 1399)
      Min: 0.0
      Max: 17.733333333333334
      Has NaN: False
      All NaN: False
    Statistics variables: ['p_value', 'p_value_fdr_corrected']

  Loading JJA...
    Departure variables: ['PRECIP_PERIOD']
    Departure dims: {'south_north': 899, 'west_east': 1399}
    Percentage varia

In [6]:
# PLOTTING FUNCTIONS

def create_map(ax, cmap, vmin, vmax, data, mwu_result=None, ax_loc_index=0):
    """Create individual map subplot"""
    
    # Subplot label dictionary
    subplot_ltr_dict = {i: chr(ord('a') + i) for i in range(26)}
    
    # Set map extent
    extent = [-125, -73, 24, 50]
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    
    # Add map features
    countries_shp = shpreader.natural_earth(resolution='50m', category='cultural', 
                                            name='admin_0_countries')
    
    for country, info in zip(shpreader.Reader(countries_shp).geometries(), 
                            shpreader.Reader(countries_shp).records()):
        if info.attributes['NAME_LONG'] != 'United States':
            ax.add_geometries([country], ccrs.PlateCarree(),
                             facecolor='Grey', edgecolor='k', zorder=3)
    
    # Add geographic features
    states_provinces = cfeature.NaturalEarthFeature(
        category='cultural', name='admin_1_states_provinces_lines', 
        scale='50m', facecolor='none')
    ax.add_feature(states_provinces, edgecolor='gray', lw=1., zorder=2)
    ax.add_feature(cfeature.LAND, facecolor=cfeature.COLORS['land_alt1'], zorder=1)
    ax.add_feature(cfeature.OCEAN, zorder=3)
    ax.add_feature(cfeature.COASTLINE, edgecolor='k', lw=.9, alpha=0.8, zorder=2)
    ax.add_feature(cfeature.BORDERS, edgecolor='dimgray', lw=1.2, zorder=2)
    ax.add_feature(cfeature.LAKES, edgecolor='k', lw=.5, zorder=3)
    
    # Extract the data variable - handle both Dataset and DataArray
    if isinstance(data, xr.Dataset):
        # Get the first data variable (should be PRECIP_PERIOD)
        var_name = list(data.data_vars)[0]
        data_to_plot = data[var_name]
    else:
        data_to_plot = data
    
    # Downsample data for plotting
    data_ds = data_to_plot.to_dataset(name='plot_var') if isinstance(data_to_plot, xr.DataArray) else data_to_plot
    data_downsampled = downsample_ds(data_ds, downsampling_factor, "mean")
    plot_data = data_downsampled['plot_var' if 'plot_var' in data_downsampled else list(data_downsampled.data_vars)[0]].squeeze()
    
    # Get coordinates - use from the data itself
    if 'lon' in data_downsampled.coords and 'lat' in data_downsampled.coords:
        lon_coords = data_downsampled.lon
        lat_coords = data_downsampled.lat
    else:
        # Fallback: downsample from hist_means
        coords_downsampled = downsample_ds(hist_means['annual'], downsampling_factor, "mean")
        lon_coords = coords_downsampled.lon
        lat_coords = coords_downsampled.lat
    
    # Plot data
    mesh = ax.pcolormesh(lon_coords, lat_coords, plot_data,
                        cmap=cmap, transform=ccrs.PlateCarree(),
                        shading='auto', vmin=vmin, vmax=vmax, zorder=1)
    
    # Add statistical significance stippling if provided
    if mwu_result is not None:
        # Create binary significance mask
        sig_mask_binary = (mwu_result.p_value_fdr_corrected < alpha)
        
        # Wrap in Dataset
        sig_mask_ds = xr.Dataset({'p_value_fdr_corrected': sig_mask_binary})
        
        # Downsample the mask - mean gives fraction of significant cells
        stat_sig_fraction = downsample_ds(sig_mask_ds, downsampling_factor, "mean")
        
        # Apply threshold
        stipple_mask = (stat_sig_fraction.p_value_fdr_corrected >= sig_display_threshold)
        
        # Add stippling with simple dots
        ax.scatter(x=lon_coords.where(stipple_mask == True, np.nan),
                   y=lat_coords.where(stipple_mask == True, np.nan),
                   color="k", s=1.0, marker='.', linewidths=0,
                   transform=ccrs.PlateCarree(), zorder=2)
    
    # Add formatting
    ax.xaxis.set_major_formatter(LONGITUDE_FORMATTER) 
    ax.yaxis.set_major_formatter(LATITUDE_FORMATTER)
    
    # Add subplot label
    anchored_text = AnchoredText(str(subplot_ltr_dict[ax_loc_index]), 
                                loc='upper left', pad=0.3, borderpad=0.2)
    ax.add_artist(anchored_text)
    
    return mesh



In [7]:
# CREATE FIGURE

print("CREATING MULTIPANEL FIGURE")

print("\nSetting up colormaps...")

# Historical data (blues/greens)
vmax0, vmin0, c_bins0 = 24.0, 0, 10
cmap0 = discrete_cmap(c_bins0, truncate_colormap(cmap=colormaps['PuBuGn'], 
                                                  minval=.2, maxval=1, n=100))
new_cmap0, tick_positions0 = create_discretized_colormap_with_arrows_and_ticks(
    cmap0, c_bins0, vmin0, vmax0)

# Seasonal historical data
vmax1, vmin1, c_bins1 = 8.0, 0, 10
cmap1 = discrete_cmap(c_bins1, truncate_colormap(cmap=colormaps['PuBuGn'], 
                                                  minval=.2, maxval=1, n=100))
new_cmap1, tick_positions1 = create_discretized_colormap_with_arrows_and_ticks(
    cmap1, c_bins1, vmin1, vmax1)

# Departure data (diverging)
vmax2, vmin2, c_bins2 = 6, -2, 6
cmap2 = discrete_cmap(c_bins2, truncate_colormap(cmap=colormaps['BrBG'], 
                                                  minval=.4, maxval=.9, n=100))
new_cmap2, tick_positions2 = create_discretized_colormap_with_arrows_and_ticks(
    cmap2, c_bins2, vmin2, vmax2)

# Seasonal departure data
vmax3, vmin3, c_bins3 = 6.0, -6.0, 10
cmap3 = discrete_cmap(c_bins3, truncate_colormap(cmap=colormaps['BrBG'], 
                                                  minval=.1, maxval=.9, n=100))
new_cmap3, tick_positions3 = create_discretized_colormap_with_arrows_and_ticks(
    cmap3, c_bins3, vmin3, vmax3)

# Percentage data
vmax4, vmin4, c_bins4 = 200, -100, 8
cmap4 = discrete_cmap(c_bins4, truncate_colormap(cmap=colormaps['PiYG'], 
                                                  minval=.25, maxval=1, n=100))
new_cmap4, tick_positions4 = create_discretized_colormap_with_arrows_and_ticks(
    cmap4, c_bins4, vmin4, vmax4)

# Seasonal percentage data
vmax5, vmin5, c_bins5 = 200, -100, 8 
cmap5 = discrete_cmap(c_bins5, truncate_colormap(cmap=colormaps['PiYG'], 
                                                  minval=.25, maxval=1, n=100))
new_cmap5, tick_positions5 = create_discretized_colormap_with_arrows_and_ticks(
    cmap5, c_bins5, vmin5, vmax5)

print("Organizing plot data...")

# Organize data for plotting
datasets = [
    hist_means['annual'],
    departures['annual'],
    percentages['annual'],
    hist_means['mam'],
    departures['mam'],
    percentages['mam'],
    hist_means['jja'],
    departures['jja'],
    percentages['jja'],
    hist_means['son'],
    departures['son'],
    percentages['son'],
    hist_means['djf'],
    departures['djf'],
    percentages['djf']
]

# Statistical tests (for stippling)
stats_for_plots = [
    None,  # hist annual
    statistics['annual'],  # departure annual
    None,  # percentage annual
    None,  # hist mam
    statistics['mam'],  # departure mam
    None,  # percentage mam
    None,  # hist jja
    statistics['jja'],  # departure jja
    None,  # percentage jja
    None,  # hist son
    statistics['son'],  # departure son
    None,  # percentage son
    None,  # hist djf
    statistics['djf'],  # departure djf
    None   # percentage djf
]

cmaps = [new_cmap0, new_cmap2, new_cmap4,
         new_cmap1, new_cmap3, new_cmap5,
         new_cmap1, new_cmap3, new_cmap5,
         new_cmap1, new_cmap3, new_cmap5,
         new_cmap1, new_cmap3, new_cmap5]

vmaxs = [vmax0, vmax2, vmax4, vmax1, vmax3, vmax5,
         vmax1, vmax3, vmax5, vmax1, vmax3, vmax5, vmax1, vmax3, vmax5]

vmins = [vmin0, vmin2, vmin4, vmin1, vmin3, vmin5,
         vmin1, vmin3, vmin5, vmin1, vmin3, vmin5, vmin1, vmin3, vmin5]

print("Setting up figure layout...")

# Set up figure and gridspec
gs = gridspec.GridSpec(8, 3, width_ratios=[4, 4, 4], 
                      height_ratios=[4, .4, 1, 4, 4, 4, 4, .4],
                      top=0.75, bottom=0.1, wspace=0.15, hspace=0.15)

fig = plt.figure(figsize=(10, 14), frameon=True, dpi=600.0)

# Create projection
p = ccrs.LambertConformal(central_longitude=-97.5, central_latitude=38.5, 
                          standard_parallels=(38.5, 38.5))

print("Creating subplot axes...")

# Create map axes
map_axes = []
colorbar_axes = []

# Row 0 - Annual
map_axes.extend([fig.add_subplot(gs[0,0], projection=p),
                 fig.add_subplot(gs[0,1], projection=p),
                 fig.add_subplot(gs[0,2], projection=p)])

colorbar_axes.extend([fig.add_subplot(gs[1,0]),
                      fig.add_subplot(gs[1,1]),
                      fig.add_subplot(gs[1,2])])

# Rows 3-6 - Seasonal
for row in [3, 4, 5, 6]:
    map_axes.extend([fig.add_subplot(gs[row,0], projection=p),
                     fig.add_subplot(gs[row,1], projection=p),
                     fig.add_subplot(gs[row,2], projection=p)])

# Last row colorbars
colorbar_axes.extend([fig.add_subplot(gs[7,0]),
                      fig.add_subplot(gs[7,1]),
                      fig.add_subplot(gs[7,2])])

# Set extent for all map axes
extent = [-125, -73, 24, 50]
for ax in map_axes:
    ax.set_extent(extent, crs=ccrs.PlateCarree())

print(f"Created {len(map_axes)} map axes and {len(colorbar_axes)} colorbar axes")

# Create maps
print("\nCreating individual maps...")
plot_maps = []

for i, (ax, ds, stats, cmap, vn, vx) in enumerate(
    zip(map_axes, datasets, stats_for_plots, cmaps, vmins, vmaxs)):
    
    print(f"  Map {i+1}/{len(map_axes)}")
    plot_map = create_map(ax=ax, cmap=cmap, vmin=vn, vmax=vx, 
                         data=ds, mwu_result=stats, ax_loc_index=i)
    plot_maps.append(plot_map)

print("\nAdding colorbars...")

# Annual colorbars
color_bar10 = plt.colorbar(plot_maps[0], orientation='horizontal', cax=colorbar_axes[0], 
                          extend='max', ticks=tick_positions0[::2], fraction=.8, shrink=.1)
color_bar10.set_label(r'count', labelpad=3, fontsize=8)
color_bar10.ax.set_position(color_bar10.ax.get_position().translated(0, 0.01))

color_bar11 = plt.colorbar(plot_maps[1], orientation='horizontal', cax=colorbar_axes[1], 
                          extend='both', ticks=tick_positions2[::2], fraction=.8, shrink=.1)
color_bar11.set_label(r'count Δ', labelpad=3, fontsize=8)
color_bar11.ax.set_position(color_bar11.ax.get_position().translated(0, 0.01))

color_bar12 = plt.colorbar(plot_maps[2], orientation='horizontal', cax=colorbar_axes[2], 
                          extend='max', ticks=tick_positions4[::2], fraction=.8, shrink=.1)
color_bar12.set_label(r'count Δ (%)', labelpad=3, fontsize=8)
color_bar12.ax.set_position(color_bar12.ax.get_position().translated(0, 0.01))

# Seasonal colorbars
color_bar70 = plt.colorbar(plot_maps[3], orientation='horizontal', cax=colorbar_axes[3], 
                          extend='max', ticks=tick_positions1[::2], fraction=.8, shrink=.1)
color_bar70.set_label(r'count', labelpad=3, fontsize=8)
color_bar70.ax.set_position(color_bar70.ax.get_position().translated(0, 0.01))

color_bar71 = plt.colorbar(plot_maps[4], orientation='horizontal', cax=colorbar_axes[4], 
                          extend='both', ticks=tick_positions3[::2], fraction=.8, shrink=.1)
color_bar71.set_label(r'count Δ', labelpad=3, fontsize=8)
color_bar71.ax.set_position(color_bar71.ax.get_position().translated(0, 0.01))

color_bar72 = plt.colorbar(plot_maps[5], orientation='horizontal', cax=colorbar_axes[5], 
                          extend='max', ticks=tick_positions5[::2], fraction=.8, shrink=.1)
color_bar72.set_label(r'count Δ (%)', labelpad=3, fontsize=8)
color_bar72.ax.set_position(color_bar72.ax.get_position().translated(0, 0.01))

print("Adding titles and labels...")

RCP_label = RCP_scenario.replace("p",".")

# Add titles
map_axes[0].title.set_text(f'HIST Annual Count\n99th percentile occurrences')
map_axes[1].title.set_text(f'EOC{RCP_label} - HIST Annual Count\n99th percentile occurrences')
map_axes[2].title.set_text(f'EOC{RCP_label} - HIST Percentage\n99th percentile occurrences')

map_axes[3].title.set_text(f'HIST Seasonal Count\n99th percentile occurrences')
map_axes[4].title.set_text(f'EOC{RCP_label} - HIST Seasonal Count\n99th percentile occurrences')
map_axes[5].title.set_text(f'EOC{RCP_label} - HIST Percentage\n99th percentile occurrences')

# Add season labels
seasons_labels = ['Annual', 'MAM', 'JJA', 'SON', 'DJF']
season_axes = [map_axes[0], map_axes[3], map_axes[6], map_axes[9], map_axes[12]]

for ax, season in zip(season_axes, seasons_labels):
    ax.text(-0.07, 0.55, season, va='bottom', ha='center',
            rotation='vertical', rotation_mode='anchor', transform=ax.transAxes)

print("Finalizing layout...")
plt.tight_layout()


CREATING MULTIPANEL FIGURE

Setting up colormaps...
Organizing plot data...
Setting up figure layout...
Creating subplot axes...
Created 15 map axes and 6 colorbar axes

Creating individual maps...
  Map 1/15
  Map 2/15
  Map 3/15
  Map 4/15
  Map 5/15
  Map 6/15
  Map 7/15
  Map 8/15
  Map 9/15
  Map 10/15
  Map 11/15
  Map 12/15
  Map 13/15
  Map 14/15
  Map 15/15

Adding colorbars...
Adding titles and labels...
Finalizing layout...


/tmp/ipykernel_32013/3894750373.py:215: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


In [8]:
fig

<Figure size 6000x8400 with 21 Axes>

In [9]:

# SAVE FIGURE

print("SAVING FIGURE")

# Save as PNG

png_file = f"{output_dir}/{figure_basename}.png"
print(f"\nSaving PNG: {png_file}")
fig.savefig(png_file, dpi=600, bbox_inches='tight')

# Save as PDF
pdf_file = f"{output_dir}/{figure_basename}.pdf"
print(f"Saving PDF: {pdf_file}")
fig.savefig(pdf_file, bbox_inches='tight')

print("\n" + "="*70)
print("FIGURE GENERATION COMPLETE")
print("="*70)
print(f"\nOutputs saved:")
print(f"  {png_file}")
print(f"  {pdf_file}")
print(f"\nFigure specifications:")
print(f"  Resolution: 600 DPI")
print(f"  Size: 10×14 inches")
print(f"  Panels: 5 rows × 3 columns = 15 maps")
print(f"  Data: {percentile_band} percentile")
print(f"  Scenario: EOC{RCP_scenario} - HIST")

plt.close(fig)
print("\nDone")

SAVING FIGURE

Saving PNG: ./figures/multipanel_99th_percentile_EOC8p5.png
Saving PDF: ./figures/multipanel_99th_percentile_EOC8p5.pdf

FIGURE GENERATION COMPLETE

Outputs saved:
  ./figures/multipanel_99th_percentile_EOC8p5.png
  ./figures/multipanel_99th_percentile_EOC8p5.pdf

Figure specifications:
  Resolution: 600 DPI
  Size: 10×14 inches
  Panels: 5 rows × 3 columns = 15 maps
  Data: 99th-100th percentile
  Scenario: EOC8p5 - HIST

Done
